# 🔬 ARISE-Plus v2: DEC Component Ablation Suite

### 🎯 Objective: Discover Which Component Drives Silhouette Score Gains in DEC
This interactive Google Colab suite systematically isolates and quantifies the exact contribution of each architectural module, loss term, and latent sub-representation during the **2-Stage Spatial Potts DEC (Deep Embedding Clustering)** training period.

---
### 🧪 Ablation Matrix Evaluated:
1. **`Full_ARISE_v2`**: Grand Champion baseline (Potts MRF + OT + Dense Gram + Spatial Contrastive + Recon).
2. **`wo_Potts_MRF`**: Standard DEC without spatial neighborhood consensus ($\lambda_{spatial} = 0.0$).
3. **`wo_Sinkhorn_OT`**: DEC without Entropic Sinkhorn Optimal Transport cross-modal alignment ($\mathcal{L}_{OT} = 0$).
4. **`wo_Dense_Gram`**: DEC without intra-cell Gram matrix Frobenius alignment ($\mathcal{L}_{dense} = 0$).
5. **`wo_Spatial_Contrastive`**: DEC without spatial neighbor contrastive graph loss ($\mathcal{L}_{spatial} = 0$).
6. **`wo_Reconstruction`**: DEC without multi-head autoencoder reconstruction ($\mathcal{L}_{recon} = 0$).
7. **`Frozen_Encoder_DEC_Only`**: Feature encoders frozen during DEC; only cluster centroids are updated.

### 🧬 Sub-Embedding Silhouette Dynamics Tracked:
- `fused_joint`: Joint multimodal latent space
- `fused_rna`: Cross-attention refined RNA representation
- `fused_aux`: Auxiliary modality (ADT / ATAC) representation
- `x_sim`: Expression similarity GCN branch
- `x_dist`: Spatial distance GCN branch

---
### 🚀 Quick Start in Google Colab:
1. Ensure GPU is enabled: **`Runtime -> Change runtime type -> T4 GPU`**.
2. Run **Step 1** to install dependencies.
3. Run **Step 2** to initialize the ablation engine.
4. Set parameters and execute in **Step 3**.
5. Inspect summary metrics & Silhouette Delta tables in **Step 4**.
6. View comparative trajectory and dynamics charts in **Step 5**.
7. Download results archive in **Step 6**.

In [ ]:
# @title 📦 Step 1: Install Dependencies (PyG, Scanpy, gdown, scikit-learn)
!pip install -q scanpy anndata gdown torch-geometric scikit-learn scikit-misc matplotlib seaborn pandas numpy

import torch
print("=" * 50)
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
print("=" * 50)

In [ ]:
# @title 📂 Step 2: Initialize ARISE-Plus v2 DEC Ablation Engine
ablation_code = "#!/usr/bin/env python3\n\"\"\"\n================================================================================\n  \ud83d\udd2c ARISE-Plus v2: Deep Embedding Clustering (DEC) Component Ablation Suite\n================================================================================\n\n  Purpose:\n  Systematically evaluate which loss terms, spatial consensus priors, and\n  latent sub-representations drive the Silhouette score improvement during the\n  2-Stage Spatial Potts DEC fine-tuning period.\n\n  Ablation Matrix:\n  1. Full_ARISE_v2: Grand Champion baseline (Potts MRF + OT + Dense Gram + Spatial Contrastive + Recon)\n  2. wo_Potts_MRF: Standard DEC with lambda_spatial = 0.0 (no neighborhood consensus smoothing)\n  3. wo_Sinkhorn_OT: DEC without cross-modal Sinkhorn Optimal Transport loss (L_ot = 0)\n  4. wo_Dense_Gram: DEC without relational Gram matrix Frobenius alignment (L_dense = 0)\n  5. wo_Spatial_Contrastive: DEC without spatial graph contrastive loss (L_spatial = 0)\n  6. wo_Reconstruction: DEC without multi-head autoencoder reconstruction loss (L_recon = 0)\n  7. Frozen_Encoder_DEC_Only: Feature encoders frozen during DEC; only cluster centers updated\n\n  Sub-Representation Silhouette Dynamics Tracked:\n  - fused_joint (64-dim): Joint multi-modal latent representation\n  - fused_rna   (64-dim): Cross-attention refined RNA representation\n  - fused_aux   (64-dim): Auxiliary ADT/ATAC representation\n  - x_sim       (64-dim): Expression similarity GCN branch\n  - x_dist      (64-dim): Spatial Euclidean distance GCN branch\n\n  Authors: FYDP Research Team\n  Date: 2026\n================================================================================\n\"\"\"\n\nimport os\nimport sys\nimport math\nimport time\nimport argparse\nimport random\nimport warnings\nfrom dataclasses import dataclass\nfrom typing import Dict, Tuple, List, Optional\n\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\nimport scanpy as sc\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch_geometric.data import Data\nfrom torch_geometric.nn import GCNConv\nfrom sklearn.cluster import KMeans\nfrom sklearn.neighbors import NearestNeighbors, kneighbors_graph\nfrom sklearn.metrics.pairwise import cosine_similarity\nfrom sklearn.metrics import (\n    adjusted_rand_score,\n    normalized_mutual_info_score,\n    adjusted_mutual_info_score,\n    homogeneity_score,\n    v_measure_score,\n    fowlkes_mallows_score,\n    silhouette_score\n)\n\nwarnings.filterwarnings('ignore')\n\n# ----------------------------------------------------------------------\n# 1. REPRODUCIBILITY SEEDING\n# ----------------------------------------------------------------------\n\ndef set_seed(seed: int = 42):\n    \"\"\"Set global random seed for deterministic execution.\"\"\"\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed(seed)\n        torch.cuda.manual_seed_all(seed)\n        torch.backends.cudnn.deterministic = True\n        torch.backends.cudnn.benchmark = False\n    os.environ['PYTHONHASHSEED'] = str(seed)\n\n\n# ----------------------------------------------------------------------\n# 2. DATA PREPROCESSING & DATASET REGISTRY\n# ----------------------------------------------------------------------\n\nBENCHMARK_DATASETS = [\n    (\"10x_human_lymph_node_A1\", \"https://drive.google.com/drive/folders/10z1N4MwW8Y49o8GlkYGBKVx1N7fiMuyC\"),\n    (\"10x_human_lymph_node_D1\", \"https://drive.google.com/drive/folders/1-g_Ca2XMaMXF-MisuVY-wobWDX86O6zz\"),\n    (\"Mouse_Brain_E11_S1\", \"https://drive.google.com/drive/folders/1zRwDJrYnks0LRzlAVRqPU7jE_OcStgPo\"),\n    (\"Mouse_Brain_E13_S1\", \"https://drive.google.com/drive/folders/1GOufwIRjjfcd9Bi2GKtebzKoPCg2jVud\"),\n    (\"Mouse_Brain_E15_S1\", \"https://drive.google.com/drive/folders/1rHkTL5OF5qPsEERypRGMS51SjUQ69tdD\"),\n    (\"Mouse_Brain_E18_S1\", \"https://drive.google.com/drive/folders/1Xj1LNIAY93biS6JIMKNRODn5GvtCKADB\"),\n]\n\ndef clr_normalize_each_cell(adata, inplace=True):\n    def seurat_clr(x):\n        s = np.sum(np.log1p(x[x > 0]))\n        exp = np.exp(s / len(x))\n        return np.log1p(x / exp)\n\n    if not inplace:\n        adata = adata.copy()\n\n    adata.X = np.apply_along_axis(\n        seurat_clr, 1, (adata.X.toarray() if sp.issparse(adata.X) else np.array(adata.X))\n    )\n    return adata\n\ndef pca(adata, use_reps=None, n_comps=10):\n    from sklearn.decomposition import PCA\n    pca_model = PCA(n_components=n_comps)\n    if use_reps is not None:\n        feat_pca = pca_model.fit_transform(adata.obsm[use_reps])\n    else:\n        feat_pca = pca_model.fit_transform(adata.X.toarray() if sp.issparse(adata.X) else adata.X)\n    return feat_pca\n\ndef tfidf(X):\n    idf = X.shape[0] / (X.sum(axis=0) + 1e-10)\n    if sp.issparse(X):\n        tf = X.multiply(1 / (X.sum(axis=1) + 1e-10))\n        return sp.csr_matrix(tf.multiply(idf))\n    else:\n        tf = X / (X.sum(axis=1, keepdims=True) + 1e-10)\n        return tf * idf\n\ndef preprocess_universal(adata_RNA, adata_omics2, dataset_name: str) -> Tuple[np.ndarray, np.ndarray]:\n    adata_RNA_copy = adata_RNA.copy()\n    sc.pp.filter_genes(adata_RNA_copy, min_cells=10)\n    try:\n        sc.pp.highly_variable_genes(adata_RNA_copy, flavor=\"seurat_v3\", n_top_genes=3000)\n        sc.pp.normalize_total(adata_RNA_copy, target_sum=1e4)\n        sc.pp.log1p(adata_RNA_copy)\n        sc.pp.scale(adata_RNA_copy)\n    except Exception:\n        sc.pp.normalize_total(adata_RNA_copy, target_sum=1e4)\n        sc.pp.log1p(adata_RNA_copy)\n        sc.pp.highly_variable_genes(adata_RNA_copy, flavor=\"seurat\", n_top_genes=3000)\n        sc.pp.scale(adata_RNA_copy)\n\n    RNA_expression = adata_RNA_copy[:, adata_RNA_copy.var['highly_variable']].X\n    if sp.issparse(RNA_expression):\n        RNA_expression = RNA_expression.toarray()\n\n    adata_omics2_copy = adata_omics2[adata_RNA.obs_names].copy()\n    if dataset_name.startswith(\"10x\"):\n        adata_omics2_copy = clr_normalize_each_cell(adata_omics2_copy)\n        sc.pp.scale(adata_omics2_copy)\n        omics2_expression = adata_omics2_copy.X\n    else:\n        adata_omics2_copy.X = tfidf(adata_omics2_copy.X)\n        sc.pp.normalize_per_cell(adata_omics2_copy, counts_per_cell_after=1e4)\n        sc.pp.log1p(adata_omics2_copy)\n        n_comps = min(60, adata_omics2_copy.shape[1])\n        adata_omics2_copy.obsm['feat'] = pca(adata_omics2_copy, n_comps=n_comps)\n        omics2_expression = adata_omics2_copy.obsm['feat']\n\n    if sp.issparse(omics2_expression):\n        omics2_expression = omics2_expression.toarray()\n\n    return RNA_expression, omics2_expression\n\ndef load_dataset(dataset_name: str, folder_url: str, base_data_dir: str = \"data\"):\n    base = os.path.join(base_data_dir, dataset_name)\n    os.makedirs(base, exist_ok=True)\n\n    rna_path = os.path.join(base, \"adata_RNA.h5ad\")\n    if dataset_name.startswith(\"10x\"):\n        other_path = os.path.join(base, \"adata_ADT.h5ad\")\n        annotation_path = os.path.join(base, \"annotation.csv\")\n        gt_col = \"manual-anno\"\n    else:\n        other_path = os.path.join(base, \"adata_ATAC.h5ad\")\n        annotation_path = os.path.join(base, \"anno.csv\")\n        gt_col = \"cluster\"\n\n    if not (os.path.exists(rna_path) and os.path.exists(other_path) and os.path.exists(annotation_path)):\n        print(f\"Downloading dataset files for {dataset_name}...\")\n        os.system(f'gdown --folder \"{folder_url}\" --output \"{base}\"')\n\n    adata_rna = sc.read_h5ad(rna_path)\n    adata_other = sc.read_h5ad(other_path)\n    adata_rna.var_names_make_unique()\n    adata_other.var_names_make_unique()\n\n    anno_df = pd.read_csv(annotation_path, index_col=0)\n    adata_rna.obs['ground_truth'] = anno_df[gt_col]\n    adata_other.obs['ground_truth'] = anno_df[gt_col]\n\n    rna_data, aux_data = preprocess_universal(adata_rna, adata_other, dataset_name)\n    cell_positions = adata_rna.obsm['spatial']\n    num_clusters = adata_rna.obs['ground_truth'].nunique()\n\n    return adata_rna, rna_data, aux_data, cell_positions, num_clusters\n\n\n# ----------------------------------------------------------------------\n# 3. GRAPH TOPOLOGY & BUILDER\n# ----------------------------------------------------------------------\n\ndef compute_multi_order_motif_matrix(adj_sparse: sp.csr_matrix, k1: float = 1.0, k2: float = 0.5, k3: float = 0.25) -> sp.csr_matrix:\n    A_bin = (adj_sparse > 0).astype(np.float32)\n    A2 = A_bin.dot(A_bin)\n    M3 = A2.multiply(A_bin).tocsr()\n\n    A3 = A2.dot(A_bin)\n    M4 = A3.multiply(A_bin).tocsr()\n\n    max_m3 = M3.data.max() if len(M3.data) > 0 else 1.0\n    if max_m3 == 0: max_m3 = 1.0\n    M3_norm = M3 / max_m3\n\n    max_m4 = M4.data.max() if len(M4.data) > 0 else 1.0\n    if max_m4 == 0: max_m4 = 1.0\n    M4_norm = M4 / max_m4\n\n    motif_total = k1 * adj_sparse + k2 * M3_norm + k3 * M4_norm\n    return motif_total.tocsr()\n\ndef build_multimodal_graph_v2(\n    x_rna: np.ndarray,\n    x_aux: np.ndarray,\n    cell_positions: np.ndarray,\n    device: str = 'cpu',\n    num_neighbors: int = 15\n) -> Data:\n    num_nodes = x_rna.shape[0]\n\n    # 1. Similarity Graph\n    similarity_matrix = cosine_similarity(x_rna)\n    nbrs = NearestNeighbors(n_neighbors=num_neighbors + 1, metric='cosine').fit(x_rna)\n    _, indices = nbrs.kneighbors(x_rna)\n\n    adj_sim = np.zeros_like(similarity_matrix, dtype=np.float32)\n    for i in range(num_nodes):\n        for j in indices[i][1:]:\n            adj_sim[i, j] = 1.0\n            adj_sim[j, i] = 1.0\n\n    sim_sparse = sp.csr_matrix(adj_sim)\n    motif_sparse = compute_multi_order_motif_matrix(sim_sparse, k1=1.0, k2=0.5, k3=0.25)\n\n    sim_nonzero = motif_sparse.nonzero()\n    sim_edge_index = torch.tensor(np.array(sim_nonzero), dtype=torch.long).to(device)\n    sim_edge_weight = torch.tensor(np.array(motif_sparse[sim_nonzero]).flatten(), dtype=torch.float).to(device)\n\n    # 2. Spatial Euclidean Graph\n    knn_spatial = kneighbors_graph(cell_positions, n_neighbors=num_neighbors, mode='distance', include_self=False)\n    knn_spatial = knn_spatial.maximum(knn_spatial.T)\n\n    dist_edge_index = torch.tensor(knn_spatial.nonzero(), dtype=torch.long).to(device)\n    dist_edge_weight = torch.tensor(knn_spatial.data, dtype=torch.float).to(device)\n\n    # 3. Intersection Scaffold\n    sim_edges = set(zip(sim_edge_index[0].tolist(), sim_edge_index[1].tolist()))\n    dist_edges = set(zip(dist_edge_index[0].tolist(), dist_edge_index[1].tolist()))\n    common_edges = sim_edges.intersection(dist_edges)\n\n    if len(common_edges) == 0:\n        common_edge_index = dist_edge_index\n        common_edge_weight = torch.ones(dist_edge_index.shape[1], dtype=torch.float).to(device)\n    else:\n        common_edge_index = torch.tensor(list(zip(*common_edges)), dtype=torch.long).to(device)\n        common_edge_weight = torch.ones(common_edge_index.shape[1], dtype=torch.float).to(device)\n\n    spatial_adj = torch.zeros((num_nodes, num_nodes), dtype=torch.float, device=device)\n    spatial_adj[dist_edge_index[0], dist_edge_index[1]] = 1.0\n    spatial_mask = spatial_adj.clone()\n    spatial_mask.fill_diagonal_(1.0)\n\n    x_RNA_tensor = torch.tensor(x_rna, dtype=torch.float).to(device)\n    x_ADT_tensor = torch.tensor(x_aux, dtype=torch.float).to(device)\n\n    data = Data(\n        x_RNA=x_RNA_tensor,\n        x_ADT=x_ADT_tensor,\n        sim_edge_index=sim_edge_index,\n        sim_edge_weight=sim_edge_weight,\n        dist_edge_index=dist_edge_index,\n        dist_edge_weight=dist_edge_weight,\n        common_edge_index=common_edge_index,\n        common_edge_weight=common_edge_weight\n    )\n    data.spatial_mask = spatial_mask\n    data.spatial_adj = spatial_adj\n    return data\n\n\n# ----------------------------------------------------------------------\n# 4. MODULE DEFINITIONS\n# ----------------------------------------------------------------------\n\nclass GraphMaskedCrossAttention(nn.Module):\n    def __init__(self, d_model: int = 64, n_heads: int = 4, dropout: float = 0.1):\n        super().__init__()\n        self.d_model = d_model\n        self.n_heads = n_heads\n        self.head_dim = d_model // n_heads\n        assert d_model % n_heads == 0, \"d_model must be divisible by n_heads\"\n\n        self.q_proj = nn.Linear(d_model, d_model)\n        self.k_proj = nn.Linear(d_model, d_model)\n        self.v_proj = nn.Linear(d_model, d_model)\n        self.out_proj = nn.Linear(d_model, d_model)\n        self.dropout = nn.Dropout(dropout)\n        self.layer_norm = nn.LayerNorm(d_model)\n\n    def forward(self, x_query: torch.Tensor, x_key_value: torch.Tensor, spatial_mask: torch.Tensor) -> torch.Tensor:\n        N, D = x_query.shape\n        residual = x_query\n\n        Q = self.q_proj(x_query).view(N, self.n_heads, self.head_dim).transpose(0, 1)\n        K = self.k_proj(x_key_value).view(N, self.n_heads, self.head_dim).transpose(0, 1)\n        V = self.v_proj(x_key_value).view(N, self.n_heads, self.head_dim).transpose(0, 1)\n\n        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)\n        mask_bool = (spatial_mask == 0).unsqueeze(0)\n        scores = scores.masked_fill(mask_bool, -1e9)\n\n        attn_weights = F.softmax(scores, dim=-1)\n        attn_weights = self.dropout(attn_weights)\n\n        context = torch.matmul(attn_weights, V)\n        context = context.transpose(0, 1).contiguous().view(N, D)\n        out = self.out_proj(context)\n\n        return self.layer_norm(residual + out)\n\n\nclass SinkhornOptimalTransportLoss(nn.Module):\n    def __init__(self, eps: float = 0.1, max_iter: int = 30):\n        super().__init__()\n        self.eps = eps\n        self.max_iter = max_iter\n\n    def forward(self, z_rna: torch.Tensor, z_aux: torch.Tensor) -> torch.Tensor:\n        N = z_rna.shape[0]\n        z_rna_norm = F.normalize(z_rna, p=2, dim=1)\n        z_aux_norm = F.normalize(z_aux, p=2, dim=1)\n\n        C = 1.0 - torch.mm(z_rna_norm, z_aux_norm.t())\n        mu = torch.full((N,), 1.0 / N, device=z_rna.device, dtype=z_rna.dtype)\n        nu = torch.full((N,), 1.0 / N, device=z_aux.device, dtype=z_aux.dtype)\n\n        K = torch.exp(-C / self.eps)\n        u = torch.ones(N, device=z_rna.device, dtype=z_rna.dtype)\n\n        for _ in range(self.max_iter):\n            v = nu / (torch.matmul(K.t(), u) + 1e-8)\n            u = mu / (torch.matmul(K, v) + 1e-8)\n\n        T = u.unsqueeze(1) * K * v.unsqueeze(0)\n        return torch.sum(T * C)\n\n\nclass SpatialPottsDEC(nn.Module):\n    def __init__(self, num_clusters: int, latent_dim: int, alpha: float = 1.0, lambda_spatial: float = 0.4):\n        super().__init__()\n        self.num_clusters = num_clusters\n        self.latent_dim = latent_dim\n        self.alpha = alpha\n        self.lambda_spatial = lambda_spatial\n        self.cluster_centers = nn.Parameter(torch.Tensor(num_clusters, latent_dim))\n        nn.init.xavier_uniform_(self.cluster_centers)\n\n    def compute_q(self, z: torch.Tensor) -> torch.Tensor:\n        dist = torch.sum((z.unsqueeze(1) - self.cluster_centers.unsqueeze(0)) ** 2, dim=2)\n        q = 1.0 / (1.0 + dist / self.alpha)\n        q = q ** ((self.alpha + 1.0) / 2.0)\n        q = q / torch.sum(q, dim=1, keepdim=True)\n        return q\n\n    def compute_spatial_target_p(self, q: torch.Tensor, spatial_adj: torch.Tensor) -> torch.Tensor:\n        weight = q ** 2 / (torch.sum(q, dim=0, keepdim=True) + 1e-8)\n        p_dec = weight / (torch.sum(weight, dim=1, keepdim=True) + 1e-8)\n\n        if self.lambda_spatial <= 1e-6:\n            return p_dec\n\n        adj_norm = spatial_adj / (torch.sum(spatial_adj, dim=1, keepdim=True) + 1e-8)\n        spatial_consensus = torch.mm(adj_norm, q)\n\n        p_spatial = p_dec * torch.exp(self.lambda_spatial * spatial_consensus)\n        p_final = p_spatial / (torch.sum(p_spatial, dim=1, keepdim=True) + 1e-8)\n        return p_final\n\n    def forward(self, z: torch.Tensor, spatial_adj: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:\n        q = self.compute_q(z)\n        p = self.compute_spatial_target_p(q.detach(), spatial_adj)\n        kl_loss = F.kl_div(q.log(), p, reduction='batchmean')\n        return q, kl_loss\n\n\n# ----------------------------------------------------------------------\n# 5. ABLATION CONFIGURATION DATACLASS\n# ----------------------------------------------------------------------\n\n@dataclass\nclass AblationConfig:\n    name: str\n    description: str\n    lambda_spatial: float = 0.4\n    enable_ot: bool = True\n    enable_dense_gram: bool = True\n    enable_spatial_contrastive: bool = True\n    enable_recon_in_dec: bool = True\n    freeze_encoders_in_dec: bool = False\n\n# Standard Ablation Registry\nABLATION_PRESETS: Dict[str, AblationConfig] = {\n    'Full_ARISE_v2': AblationConfig(\n        name='Full_ARISE_v2',\n        description='Full Grand Champion Architecture (Potts MRF + OT + Dense Gram + Spatial + Recon)',\n        lambda_spatial=0.4,\n        enable_ot=True,\n        enable_dense_gram=True,\n        enable_spatial_contrastive=True,\n        enable_recon_in_dec=True,\n        freeze_encoders_in_dec=False\n    ),\n    'wo_Potts_MRF': AblationConfig(\n        name='wo_Potts_MRF',\n        description='Standard DEC without Potts MRF Spatial Smoothing (lambda_spatial=0.0)',\n        lambda_spatial=0.0,\n        enable_ot=True,\n        enable_dense_gram=True,\n        enable_spatial_contrastive=True,\n        enable_recon_in_dec=True,\n        freeze_encoders_in_dec=False\n    ),\n    'wo_Sinkhorn_OT': AblationConfig(\n        name='wo_Sinkhorn_OT',\n        description='DEC without Entropic Sinkhorn Optimal Transport Loss (L_ot=0 in DEC)',\n        lambda_spatial=0.4,\n        enable_ot=False,\n        enable_dense_gram=True,\n        enable_spatial_contrastive=True,\n        enable_recon_in_dec=True,\n        freeze_encoders_in_dec=False\n    ),\n    'wo_Dense_Gram': AblationConfig(\n        name='wo_Dense_Gram',\n        description='DEC without Dense Relational Gram Matrix Frobenius Alignment (L_dense=0 in DEC)',\n        lambda_spatial=0.4,\n        enable_ot=True,\n        enable_dense_gram=False,\n        enable_spatial_contrastive=True,\n        enable_recon_in_dec=True,\n        freeze_encoders_in_dec=False\n    ),\n    'wo_Spatial_Contrastive': AblationConfig(\n        name='wo_Spatial_Contrastive',\n        description='DEC without Spatial Graph Contrastive Loss (L_spatial=0 in DEC)',\n        lambda_spatial=0.4,\n        enable_ot=True,\n        enable_dense_gram=True,\n        enable_spatial_contrastive=False,\n        enable_recon_in_dec=True,\n        freeze_encoders_in_dec=False\n    ),\n    'wo_Reconstruction': AblationConfig(\n        name='wo_Reconstruction',\n        description='DEC without Multi-Head Reconstruction Loss (L_recon=0 in DEC)',\n        lambda_spatial=0.4,\n        enable_ot=True,\n        enable_dense_gram=True,\n        enable_spatial_contrastive=True,\n        enable_recon_in_dec=False,\n        freeze_encoders_in_dec=False\n    ),\n    'Frozen_Encoder_DEC_Only': AblationConfig(\n        name='Frozen_Encoder_DEC_Only',\n        description='Encoders frozen during DEC; only cluster centers updated',\n        lambda_spatial=0.4,\n        enable_ot=True,\n        enable_dense_gram=True,\n        enable_spatial_contrastive=True,\n        enable_recon_in_dec=True,\n        freeze_encoders_in_dec=True\n    )\n}\n\n\n# ----------------------------------------------------------------------\n# 6. MODEL ARCHITECTURE WITH ABLATION CONTROLS\n# ----------------------------------------------------------------------\n\nclass ARISEPlusV2AblationModel(nn.Module):\n    def __init__(\n        self,\n        in_rna_dim: int,\n        in_aux_dim: int,\n        num_clusters: int,\n        hidden_dim: int = 512,\n        out_dim: int = 64,\n        lambda_spatial: float = 0.4\n    ):\n        super().__init__()\n\n        # Graph Encoders\n        self.x_RNA1 = GCNConv(in_rna_dim, hidden_dim)\n        self.x_RNA2 = GCNConv(in_rna_dim, hidden_dim)\n        self.sim_conv = GCNConv(hidden_dim, out_dim)\n        self.dist_conv = GCNConv(hidden_dim, out_dim)\n        self.aux_conv = GCNConv(in_aux_dim, out_dim)\n\n        # Intra-Omic Fusion\n        self.fusion1 = nn.Sequential(nn.Linear(2 * out_dim, out_dim))\n\n        # Spatial Cross-Attention\n        self.cross_attn = GraphMaskedCrossAttention(d_model=out_dim, n_heads=4, dropout=0.1)\n\n        # Joint Inter-Omic Fusion\n        self.fusion2 = nn.Sequential(nn.Linear(2 * out_dim, out_dim))\n\n        # Decoders\n        self.deconv1 = nn.Linear(out_dim, hidden_dim)\n        self.deconv_rna = nn.Linear(hidden_dim, in_rna_dim)\n        self.deconv_aux = nn.Linear(hidden_dim, in_aux_dim)\n        self.deconv_joint = nn.Linear(hidden_dim, in_rna_dim + in_aux_dim)\n\n        # Objectives\n        self.ot_loss = SinkhornOptimalTransportLoss(eps=0.1, max_iter=30)\n        self.spatial_potts_dec = SpatialPottsDEC(num_clusters=num_clusters, latent_dim=out_dim, lambda_spatial=lambda_spatial)\n\n        # Homoscedastic Multi-Task Uncertainty Parameters (s_0..s_4)\n        self.log_vars = nn.Parameter(torch.zeros(5))\n\n    def set_cluster_centers(self, centers_np: np.ndarray):\n        dev = next(self.parameters()).device\n        self.spatial_potts_dec.cluster_centers.data = torch.tensor(centers_np, dtype=torch.float, device=dev)\n\n    def freeze_encoders(self):\n        for name, param in self.named_parameters():\n            if 'spatial_potts_dec' not in name:\n                param.requires_grad = False\n\n    def unfreeze_all(self):\n        for param in self.parameters():\n            param.requires_grad = True\n\n    def forward(self, data: Data, compute_q: bool = False) -> Dict[str, torch.Tensor]:\n        xs = F.relu(self.x_RNA1(data.x_RNA, data.sim_edge_index, data.sim_edge_weight))\n        x_sim = self.sim_conv(xs, data.sim_edge_index, data.sim_edge_weight)\n\n        xd = F.relu(self.x_RNA2(data.x_RNA, data.dist_edge_index, data.dist_edge_weight))\n        x_dist = self.dist_conv(xd, data.dist_edge_index, data.dist_edge_weight)\n\n        x_aux = self.aux_conv(data.x_ADT, data.common_edge_index, data.common_edge_weight)\n\n        z_rna = self.fusion1(torch.cat([x_sim, x_dist], dim=1))\n        z_rna_refined = self.cross_attn(z_rna, x_aux, data.spatial_mask)\n        fused_joint = self.fusion2(torch.cat([z_rna_refined, x_aux], dim=1))\n\n        q, kl_loss = None, None\n        if compute_q:\n            q, kl_loss = self.spatial_potts_dec(fused_joint, data.spatial_adj)\n\n        return {\n            'x_sim': x_sim,\n            'x_dist': x_dist,\n            'fused_rna': z_rna_refined,\n            'fused_aux': x_aux,\n            'x_aux': x_aux,\n            'fused_joint': fused_joint,\n            'embedding': fused_joint,\n            'q': q,\n            'kl_loss': kl_loss\n        }\n\n    def compute_loss(\n        self,\n        data: Data,\n        outputs: Dict[str, torch.Tensor],\n        stage: int = 1,\n        cfg: Optional[AblationConfig] = None\n    ) -> Tuple[torch.Tensor, Dict[str, float]]:\n        num_nodes = data.x_RNA.shape[0]\n\n        # 1. Multi-Head Reconstruction Loss\n        l_rec = F.mse_loss(torch.cat([data.x_RNA, data.x_ADT], dim=1), self.deconv_joint(F.relu(self.deconv1(outputs['fused_joint']))))\n        l_sim = F.mse_loss(data.x_RNA, self.deconv_rna(F.relu(self.deconv1(outputs['x_sim']))))\n        l_dist = F.mse_loss(data.x_RNA, self.deconv_rna(F.relu(self.deconv1(outputs['x_dist']))))\n        l_aux = F.mse_loss(data.x_ADT, self.deconv_aux(F.relu(self.deconv1(outputs['x_aux']))))\n        total_recon = l_rec + l_sim + l_dist + l_aux\n\n        # 2. Spatial Regularization Contrastive Loss\n        graph_nei = data.spatial_adj\n        graph_neg = 1.0 - graph_nei\n        emb_norm = F.normalize(outputs['fused_rna'], p=2, dim=1, eps=1e-8)\n        sim_mat = torch.sigmoid(torch.matmul(emb_norm, emb_norm.T) - torch.diag_embed(torch.diag(torch.matmul(emb_norm, emb_norm.T))))\n        neigh_loss = torch.mul(graph_nei, torch.log(sim_mat + 1e-10)).mean()\n        neg_loss = torch.mul(graph_neg, torch.log(1.0 - sim_mat + 1e-10)).mean()\n        l_spatial = -(neigh_loss + neg_loss) / 2.0\n\n        # 3. Dense Relational Gram Alignment Loss\n        l_emb = torch.mean((outputs['fused_joint'] - outputs['fused_rna']) ** 2) + torch.mean((outputs['fused_joint'] - outputs['fused_aux']) ** 2)\n        gram_r = torch.mm(outputs['fused_rna'], outputs['fused_rna'].t())\n        gram_a = torch.mm(outputs['fused_aux'], outputs['fused_aux'].t())\n        l_rel = torch.norm(gram_r - gram_a, p='fro') / (num_nodes * num_nodes)\n        l_dense = l_emb + 0.1 * l_rel\n\n        # 4. Sinkhorn Optimal Transport Loss\n        l_ot = self.ot_loss(outputs['fused_rna'], outputs['fused_aux'])\n\n        # Precision weights (exp(-s_m))\n        p0 = torch.exp(-self.log_vars[0])\n        p1 = torch.exp(-self.log_vars[1])\n        p2 = torch.exp(-self.log_vars[2])\n        p3 = torch.exp(-self.log_vars[3])\n        p4 = torch.exp(-self.log_vars[4])\n\n        if stage == 1:\n            total_loss = (0.5 * p0 * total_recon + 0.5 * self.log_vars[0] +\n                          0.5 * p1 * l_spatial + 0.5 * self.log_vars[1] +\n                          0.5 * p2 * l_dense + 0.5 * self.log_vars[2] +\n                          0.5 * p3 * l_ot + 0.5 * self.log_vars[3])\n        else:\n            # Stage 2: Selective loss inclusion based on AblationConfig\n            total_loss = 0.0\n            if cfg is None or cfg.enable_recon_in_dec:\n                total_loss += 0.5 * p0 * total_recon + 0.5 * self.log_vars[0]\n            if cfg is None or cfg.enable_spatial_contrastive:\n                total_loss += 0.5 * p1 * l_spatial + 0.5 * self.log_vars[1]\n            if cfg is None or cfg.enable_dense_gram:\n                total_loss += 0.5 * p2 * l_dense + 0.5 * self.log_vars[2]\n            if cfg is None or cfg.enable_ot:\n                total_loss += 0.5 * p3 * l_ot + 0.5 * self.log_vars[3]\n\n            if outputs['kl_loss'] is not None:\n                l_kl = outputs['kl_loss']\n                total_loss += 0.5 * p4 * l_kl + 0.5 * self.log_vars[4]\n\n        loss_dict = {\n            'loss_total': float(total_loss.item() if isinstance(total_loss, torch.Tensor) else total_loss),\n            'loss_recon': float(total_recon.item()),\n            'loss_spatial': float(l_spatial.item()),\n            'loss_dense': float(l_dense.item()),\n            'loss_ot': float(l_ot.item()),\n            'loss_kl': float(outputs['kl_loss'].item() if outputs['kl_loss'] is not None else 0.0)\n        }\n        return total_loss, loss_dict\n\n\n# ----------------------------------------------------------------------\n# 7. EVALUATION & METRICS COMPUTATION\n# ----------------------------------------------------------------------\n\ndef compute_all_metrics(y_true: np.ndarray, y_pred: np.ndarray, embeddings: Optional[np.ndarray] = None) -> Dict[str, float]:\n    y_true_str = np.asarray(y_true).astype(str)\n    y_pred_str = np.asarray(y_pred).astype(str)\n\n    ari = float(adjusted_rand_score(y_true_str, y_pred_str))\n    nmi = float(normalized_mutual_info_score(y_true_str, y_pred_str))\n    ami = float(adjusted_mutual_info_score(y_true_str, y_pred_str))\n    homo = float(homogeneity_score(y_true_str, y_pred_str))\n    v_meas = float(v_measure_score(y_true_str, y_pred_str))\n    fmi = float(fowlkes_mallows_score(y_true_str, y_pred_str))\n\n    sil = 0.0\n    if embeddings is not None and len(np.unique(y_pred_str)) > 1:\n        try:\n            sil = float(silhouette_score(embeddings, y_pred_str))\n        except Exception:\n            sil = 0.0\n\n    return {\n        'ARI': ari,\n        'NMI': nmi,\n        'AMI': ami,\n        'Homogeneity': homo,\n        'V-measure': v_meas,\n        'FMI': fmi,\n        'Silhouette': sil\n    }\n\ndef run_kmeans_clustering(embeddings: np.ndarray, num_clusters: int, seed: int = 42) -> np.ndarray:\n    kmeans = KMeans(n_clusters=num_clusters, n_init=10, random_state=seed)\n    return kmeans.fit_predict(embeddings)\n\n\n# ----------------------------------------------------------------------\n# 8. TRAINING & ABLATION EXECUTION ENGINE\n# ----------------------------------------------------------------------\n\ndef run_dec_ablation_experiment(\n    cfg: AblationConfig,\n    data: Data,\n    ground_truth: np.ndarray,\n    num_clusters: int,\n    epochs: int = 400,\n    pretrain_epochs: Optional[int] = None,\n    finetune_epochs: Optional[int] = None,\n    lr: float = 0.001,\n    seed: int = 42,\n    device: str = 'cuda',\n    verbose: bool = True\n) -> Tuple[Dict[str, float], pd.DataFrame, pd.DataFrame]:\n    \"\"\"\n    Executes a single ablation experiment, logging epoch-by-epoch Silhouette dynamics\n    for both the primary cluster metric and the internal sub-embeddings during DEC.\n    \"\"\"\n    start_time = time.time()\n    set_seed(seed)\n    dev = torch.device(device if torch.cuda.is_available() and device == 'cuda' else 'cpu')\n    data = data.to(dev)\n\n    in_rna_dim = data.x_RNA.shape[1]\n    in_aux_dim = data.x_ADT.shape[1]\n\n    if pretrain_epochs is None and finetune_epochs is None:\n        if epochs == 400:\n            pretrain_epochs = 250\n            finetune_epochs = 150\n        else:\n            pretrain_epochs = int(epochs * 0.625)\n            finetune_epochs = epochs - pretrain_epochs\n    elif pretrain_epochs is not None and finetune_epochs is None:\n        finetune_epochs = max(epochs - pretrain_epochs, 50)\n    elif finetune_epochs is not None and pretrain_epochs is None:\n        pretrain_epochs = max(epochs - finetune_epochs, 100)\n\n    model = ARISEPlusV2AblationModel(\n        in_rna_dim=in_rna_dim,\n        in_aux_dim=in_aux_dim,\n        num_clusters=num_clusters,\n        hidden_dim=512,\n        out_dim=64,\n        lambda_spatial=cfg.lambda_spatial\n    ).to(dev)\n\n    optimizer = torch.optim.Adam(model.parameters(), lr=lr)\n\n    # STAGE 1: Standard Pre-training (identical across ablations for fair DEC evaluation)\n    best_pretrain_emb = None\n    best_pretrain_sil = -1.0\n    best_pretrain_labels = None\n\n    model.train()\n    for epoch in range(pretrain_epochs):\n        optimizer.zero_grad()\n        outputs = model(data, compute_q=False)\n        loss, _ = model.compute_loss(data, outputs, stage=1)\n        loss.backward()\n        optimizer.step()\n\n        model.eval()\n        with torch.no_grad():\n            eval_out = model(data, compute_q=False)\n            emb = eval_out['embedding'].detach().cpu().numpy()\n        model.train()\n\n        pred_labels = run_kmeans_clustering(emb, num_clusters, seed=seed)\n        metrics = compute_all_metrics(ground_truth, pred_labels, emb)\n        sil = metrics['Silhouette']\n\n        if sil > best_pretrain_sil or best_pretrain_emb is None:\n            best_pretrain_sil = sil\n            best_pretrain_emb = emb.copy()\n            best_pretrain_labels = pred_labels.copy()\n\n    # Initial Cluster Prototype Setup from Pre-trained Space\n    kmeans = KMeans(n_clusters=num_clusters, random_state=seed, n_init=20).fit(best_pretrain_emb)\n    model.set_cluster_centers(kmeans.cluster_centers_)\n\n    # Measure Initial (Epoch 0 of DEC) State\n    initial_dec_sil = best_pretrain_sil\n    initial_dec_metrics = compute_all_metrics(ground_truth, best_pretrain_labels, best_pretrain_emb)\n\n    # Freeze encoders if configured\n    if cfg.freeze_encoders_in_dec:\n        model.freeze_encoders()\n        optimizer = torch.optim.Adam([model.spatial_potts_dec.cluster_centers], lr=lr)\n\n    # STAGE 2: DEC Fine-Tuning with Ablation Controls & Fine-Grained Logging\n    dec_epoch_records = []\n    subcomp_records = []\n\n    best_sil = -1.0\n    best_ari_val = -1.0\n    best_dec_epoch = 0\n    best_embeddings = None\n    best_labels = None\n\n    model.train()\n    for epoch in range(finetune_epochs):\n        optimizer.zero_grad()\n        outputs = model(data, compute_q=True)\n        loss, loss_dict = model.compute_loss(data, outputs, stage=2, cfg=cfg)\n        loss.backward()\n        optimizer.step()\n\n        q = outputs['q'].detach()\n        pred_labels = torch.argmax(q, dim=1).cpu().numpy()\n        emb_joint = outputs['fused_joint'].detach().cpu().numpy()\n\n        metrics = compute_all_metrics(ground_truth, pred_labels, emb_joint)\n        sil = metrics['Silhouette']\n        ari = metrics['ARI']\n\n        if sil > best_sil:\n            best_sil = sil\n            best_dec_epoch = epoch + 1\n            best_embeddings = emb_joint.copy()\n            best_labels = pred_labels.copy()\n\n        if ari > best_ari_val:\n            best_ari_val = ari\n\n        # Track DEC Metric Trajectory\n        epoch_record = {\n            'ablation': cfg.name,\n            'seed': seed,\n            'dec_epoch': epoch + 1,\n            'total_epoch': pretrain_epochs + epoch + 1,\n            'loss_total': loss_dict['loss_total'],\n            'loss_kl': loss_dict['loss_kl'],\n            'loss_recon': loss_dict['loss_recon'],\n            'loss_spatial': loss_dict['loss_spatial'],\n            'loss_dense': loss_dict['loss_dense'],\n            'loss_ot': loss_dict['loss_ot'],\n            'ARI': ari,\n            'NMI': metrics['NMI'],\n            'Silhouette': sil,\n            'Homogeneity': metrics['Homogeneity']\n        }\n        dec_epoch_records.append(epoch_record)\n\n        # Track Silhouette Score across all Sub-Embedding Components\n        subcomp_names = ['fused_joint', 'fused_rna', 'fused_aux', 'x_sim', 'x_dist']\n        sub_row = {'ablation': cfg.name, 'seed': seed, 'dec_epoch': epoch + 1}\n        for cname in subcomp_names:\n            c_emb = outputs[cname].detach().cpu().numpy()\n            if len(np.unique(pred_labels)) > 1:\n                try:\n                    c_sil = float(silhouette_score(c_emb, pred_labels))\n                except Exception:\n                    c_sil = 0.0\n            else:\n                c_sil = 0.0\n            sub_row[cname] = c_sil\n        subcomp_records.append(sub_row)\n\n        if (epoch + 1) % 50 == 0 or epoch == 0 or epoch == finetune_epochs - 1:\n            if verbose:\n                print(f\"[{cfg.name} | Seed {seed}] DEC Epoch {epoch+1:3d}/{finetune_epochs} | Loss: {loss_dict['loss_total']:.4f} | KL: {loss_dict['loss_kl']:.4f} | ARI: {ari:.4f} | Sil: {sil:.4f} | \u0394Sil: {sil - initial_dec_sil:+.4f}\")\n\n    elapsed_time = time.time() - start_time\n    final_metrics = compute_all_metrics(ground_truth, best_labels, best_embeddings)\n    final_metrics['Initial_DEC_Sil'] = float(initial_dec_sil)\n    final_metrics['Initial_DEC_ARI'] = float(initial_dec_metrics['ARI'])\n    final_metrics['Peak_DEC_Sil'] = float(best_sil)\n    final_metrics['Delta_DEC_Sil'] = float(best_sil - initial_dec_sil)\n    final_metrics['Peak_DEC_ARI'] = float(best_ari_val)\n    final_metrics['Best_DEC_Epoch'] = int(best_dec_epoch)\n    final_metrics['Runtime_Sec'] = round(float(elapsed_time), 2)\n\n    df_dec_epochs = pd.DataFrame(dec_epoch_records)\n    df_subcomp = pd.DataFrame(subcomp_records)\n    return final_metrics, df_dec_epochs, df_subcomp\n\n\n# ----------------------------------------------------------------------\n# 9. VISUALIZATION ENGINE FOR ABLATION ANALYSIS\n# ----------------------------------------------------------------------\n\ndef generate_ablation_visualizations(\n    df_all_epochs: pd.DataFrame,\n    df_all_subcomp: pd.DataFrame,\n    df_summary: pd.DataFrame,\n    out_dir: str,\n    dataset_name: str\n):\n    \"\"\"\n    Produces publication-grade visual comparative trajectory charts:\n    1. Silhouette Score Trajectory across DEC Epochs (Comparison across ablations)\n    2. Sub-Embedding Component Silhouette Dynamics (Comparison across internal representations)\n    3. Net Silhouette Delta Bar Chart (Comparing component contributions)\n    \"\"\"\n    os.makedirs(out_dir, exist_ok=True)\n    sns.set_theme(style=\"whitegrid\")\n    palette = sns.color_palette(\"tab10\")\n\n    # 1. Comparative Silhouette Trajectory across Ablations\n    plt.figure(figsize=(12, 7), dpi=150)\n    sns.lineplot(\n        data=df_all_epochs,\n        x='dec_epoch',\n        y='Silhouette',\n        hue='ablation',\n        style='ablation',\n        linewidth=2.5,\n        palette=palette\n    )\n    plt.title(f\"DEC Silhouette Score Trajectory Across Ablations ({dataset_name})\", fontsize=14, fontweight='bold', pad=12)\n    plt.xlabel(\"DEC Fine-Tuning Epoch\", fontsize=12, fontweight='semibold')\n    plt.ylabel(\"Silhouette Score\", fontsize=12, fontweight='semibold')\n    plt.legend(title=\"Ablation Setting\", bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True)\n    plt.tight_layout()\n    sil_path = os.path.join(out_dir, f\"{dataset_name}_dec_ablation_silhouette_trajectory.png\")\n    plt.savefig(sil_path, bbox_inches='tight')\n    plt.close()\n    print(f\"Saved Silhouette trajectory plot to: {sil_path}\")\n\n    # 2. Comparative ARI Trajectory across Ablations\n    plt.figure(figsize=(12, 7), dpi=150)\n    sns.lineplot(\n        data=df_all_epochs,\n        x='dec_epoch',\n        y='ARI',\n        hue='ablation',\n        style='ablation',\n        linewidth=2.5,\n        palette=palette\n    )\n    plt.title(f\"DEC ARI Trajectory Across Ablations ({dataset_name})\", fontsize=14, fontweight='bold', pad=12)\n    plt.xlabel(\"DEC Fine-Tuning Epoch\", fontsize=12, fontweight='semibold')\n    plt.ylabel(\"Adjusted Rand Index (ARI)\", fontsize=12, fontweight='semibold')\n    plt.legend(title=\"Ablation Setting\", bbox_to_anchor=(1.05, 1), loc='upper left', frameon=True)\n    plt.tight_layout()\n    ari_path = os.path.join(out_dir, f\"{dataset_name}_dec_ablation_ari_trajectory.png\")\n    plt.savefig(ari_path, bbox_inches='tight')\n    plt.close()\n    print(f\"Saved ARI trajectory plot to: {ari_path}\")\n\n    # 3. Sub-Embedding Component Silhouette Dynamics (for Full_ARISE_v2 baseline)\n    df_full_sub = df_all_subcomp[df_all_subcomp['ablation'] == 'Full_ARISE_v2']\n    if not df_full_sub.empty:\n        df_melted = df_full_sub.melt(\n            id_vars=['ablation', 'seed', 'dec_epoch'],\n            value_vars=['fused_joint', 'fused_rna', 'fused_aux', 'x_sim', 'x_dist'],\n            var_name='Component',\n            value_name='Silhouette'\n        )\n        plt.figure(figsize=(11, 6), dpi=150)\n        sns.lineplot(\n            data=df_melted,\n            x='dec_epoch',\n            y='Silhouette',\n            hue='Component',\n            linewidth=2.5,\n            palette=\"Set2\"\n        )\n        plt.title(f\"Sub-Embedding Silhouette Dynamics in DEC (Full ARISE-Plus v2 | {dataset_name})\", fontsize=14, fontweight='bold', pad=12)\n        plt.xlabel(\"DEC Fine-Tuning Epoch\", fontsize=12, fontweight='semibold')\n        plt.ylabel(\"Silhouette Score of Sub-Space\", fontsize=12, fontweight='semibold')\n        plt.legend(title=\"Latent Component\", frameon=True)\n        plt.tight_layout()\n        sub_path = os.path.join(out_dir, f\"{dataset_name}_dec_subcomponent_silhouette_dynamics.png\")\n        plt.savefig(sub_path, bbox_inches='tight')\n        plt.close()\n        print(f\"Saved Sub-Component Silhouette dynamics plot to: {sub_path}\")\n\n    # 4. Net Silhouette Gain (Delta Silhouette = Peak_DEC_Sil - Initial_DEC_Sil)\n    if 'Delta_DEC_Sil' in df_summary.columns:\n        plt.figure(figsize=(10, 5), dpi=150)\n        agg_delta = df_summary.groupby('ablation')['Delta_DEC_Sil'].mean().reset_index()\n        agg_delta = agg_delta.sort_values(by='Delta_DEC_Sil', ascending=False)\n        colors = ['#10b981' if v >= 0 else '#ef4444' for v in agg_delta['Delta_DEC_Sil']]\n        sns.barplot(data=agg_delta, x='ablation', y='Delta_DEC_Sil', palette=colors)\n        plt.axhline(0, color='gray', linestyle='--', alpha=0.7)\n        plt.title(f\"Net DEC Silhouette Gain (\u0394 Silhouette) by Component ({dataset_name})\", fontsize=13, fontweight='bold', pad=12)\n        plt.xlabel(\"Ablation Setting\", fontsize=11, fontweight='semibold')\n        plt.ylabel(\"\u0394 Silhouette (Peak - Initial)\", fontsize=11, fontweight='semibold')\n        plt.xticks(rotation=25, ha='right')\n        plt.tight_layout()\n        bar_path = os.path.join(out_dir, f\"{dataset_name}_dec_delta_silhouette_comparison.png\")\n        plt.savefig(bar_path, bbox_inches='tight')\n        plt.close()\n        print(f\"Saved Delta Silhouette bar plot to: {bar_path}\")\n\n\n# ----------------------------------------------------------------------\n# 10. CLI ENTRYPOINT & ORCHESTRATION\n# ----------------------------------------------------------------------\n\ndef parse_args():\n    parser = argparse.ArgumentParser(description=\"ARISE-Plus v2: DEC Component Ablation Suite\")\n    parser.add_argument('--dataset', type=str, default='0', help=\"Dataset index (0-5), comma-separated list, or 'all'\")\n    parser.add_argument('--data_dir', type=str, default='data', help=\"Local data directory\")\n    parser.add_argument('--seeds', nargs='+', type=int, default=[42], help=\"Random seeds (default: [42])\")\n    parser.add_argument('--epochs', type=int, default=400, help=\"Total epochs (default: 400)\")\n    parser.add_argument('--pretrain_epochs', type=int, default=None, help=\"Explicit pretrain epochs (default: 250)\")\n    parser.add_argument('--finetune_epochs', type=int, default=None, help=\"Explicit DEC finetune epochs (default: 150)\")\n    parser.add_argument('--lr', type=float, default=0.001, help=\"Learning rate\")\n    parser.add_argument('--ablations', type=str, default='all', help=\"Comma-separated ablation names or 'all'\")\n    parser.add_argument('--out_dir', type=str, default='results_dec_ablation', help=\"Output directory for ablation logs and plots\")\n    parser.add_argument('--plot_only', action='store_true', help=\"Regenerate all ablation plots from saved CSVs without training\")\n    return parser.parse_args()\n\ndef main():\n    args = parse_args()\n    os.makedirs(args.out_dir, exist_ok=True)\n\n    summary_csv_path = os.path.join(args.out_dir, \"dec_ablation_summary.csv\")\n    epochs_csv_path = os.path.join(args.out_dir, \"dec_ablation_epoch_dynamics.csv\")\n    subcomp_csv_path = os.path.join(args.out_dir, \"dec_subcomponent_silhouette_dynamics.csv\")\n\n    # If --plot_only is requested, load existing CSV files and regenerate plots\n    if args.plot_only:\n        print(\"=\" * 80)\n        print(\"  \ud83d\udcca ARISE-Plus v2: Regenerating Plots from Saved Epoch Data\")\n        print(f\"  Source Directory: {args.out_dir}\")\n        print(\"=\" * 80)\n\n        if not (os.path.exists(summary_csv_path) and os.path.exists(epochs_csv_path) and os.path.exists(subcomp_csv_path)):\n            print(f\"\u274c Error: Required CSV logs not found in {args.out_dir}!\")\n            print(f\"Expected files: \\n - {summary_csv_path}\\n - {epochs_csv_path}\\n - {subcomp_csv_path}\")\n            return\n\n        df_summary = pd.read_csv(summary_csv_path)\n        df_epochs = pd.read_csv(epochs_csv_path)\n        df_subcomp = pd.read_csv(subcomp_csv_path)\n\n        datasets = df_epochs['dataset'].unique()\n        for ds_name in datasets:\n            print(f\"\\n---> Generating plots for dataset: {ds_name}\")\n            generate_ablation_visualizations(\n                df_all_epochs=df_epochs[df_epochs['dataset'] == ds_name],\n                df_all_subcomp=df_subcomp[df_subcomp['dataset'] == ds_name],\n                df_summary=df_summary[df_summary['dataset'] == ds_name],\n                out_dir=args.out_dir,\n                dataset_name=ds_name\n            )\n        print(\"\\n[Done] All plots regenerated successfully from per-epoch CSV data.\")\n        return\n\n    device = 'cuda' if torch.cuda.is_available() else 'cpu'\n\n    # Filter target datasets\n    if args.dataset == 'all':\n        target_datasets = BENCHMARK_DATASETS\n    else:\n        indices = [int(i.strip()) for i in args.dataset.split(',') if i.strip().isdigit()]\n        target_datasets = [BENCHMARK_DATASETS[i] for i in indices if i < len(BENCHMARK_DATASETS)]\n\n    # Filter target ablations\n    if args.ablations == 'all':\n        target_ablations = list(ABLATION_PRESETS.values())\n    else:\n        ablation_names = [a.strip() for a in args.ablations.split(',')]\n        target_ablations = [ABLATION_PRESETS[name] for name in ablation_names if name in ABLATION_PRESETS]\n\n    print(\"=\" * 80)\n    print(\"  \ud83d\udd2c ARISE-Plus v2: DEC Component Ablation Suite\")\n    print(f\"  Device: {device.upper()} | Seeds: {args.seeds} | Pretrain: {args.pretrain_epochs or 250} | DEC: {args.finetune_epochs or 150}\")\n    print(f\"  Target Ablations ({len(target_ablations)}): {[a.name for a in target_ablations]}\")\n    print(f\"  Output Directory: {args.out_dir}\")\n    print(\"=\" * 80)\n\n    summary_rows = []\n    all_dec_epochs_list = []\n    all_subcomp_list = []\n    total_start = time.time()\n\n    for ds_idx, (ds_name, ds_url) in enumerate(target_datasets):\n        print(f\"\\n{'='*80}\\n========== DATASET: {ds_name} ==========\\n{'='*80}\")\n        adata_rna, rna_data, aux_data, cell_positions, num_clusters = load_dataset(ds_name, ds_url, args.data_dir)\n        ground_truth = np.array(adata_rna.obs['ground_truth'].astype('category').cat.codes)\n\n        # Build Graph Data once per dataset\n        graph_data = build_multimodal_graph_v2(rna_data, aux_data, cell_positions, device=device, num_neighbors=15)\n\n        for ablation_cfg in target_ablations:\n            print(f\"\\n---> Running Ablation: [{ablation_cfg.name}] : {ablation_cfg.description}\")\n            for seed in args.seeds:\n                metrics, df_dec_ep, df_sub = run_dec_ablation_experiment(\n                    cfg=ablation_cfg,\n                    data=graph_data,\n                    ground_truth=ground_truth,\n                    num_clusters=num_clusters,\n                    epochs=args.epochs,\n                    pretrain_epochs=args.pretrain_epochs,\n                    finetune_epochs=args.finetune_epochs,\n                    lr=args.lr,\n                    seed=seed,\n                    device=device,\n                    verbose=True\n                )\n\n                df_dec_ep['dataset'] = ds_name\n                df_sub['dataset'] = ds_name\n                all_dec_epochs_list.append(df_dec_ep)\n                all_subcomp_list.append(df_sub)\n\n                res_row = {\n                    'dataset': ds_name,\n                    'ablation': ablation_cfg.name,\n                    'seed': seed,\n                    **metrics\n                }\n                summary_rows.append(res_row)\n\n        # Incrementally save and Plot per Dataset\n        df_ds_summary = pd.DataFrame(summary_rows)\n        df_ds_epochs = pd.concat(all_dec_epochs_list, ignore_index=True)\n        df_ds_subcomp = pd.concat(all_subcomp_list, ignore_index=True)\n\n        # Save per-dataset CSV files\n        ds_prefix = os.path.join(args.out_dir, ds_name)\n        df_ds_summary[df_ds_summary['dataset'] == ds_name].to_csv(f\"{ds_prefix}_dec_summary.csv\", index=False)\n        df_ds_epochs[df_ds_epochs['dataset'] == ds_name].to_csv(f\"{ds_prefix}_dec_ablation_epoch_dynamics.csv\", index=False)\n        df_ds_subcomp[df_ds_subcomp['dataset'] == ds_name].to_csv(f\"{ds_prefix}_dec_subcomponent_silhouette_dynamics.csv\", index=False)\n\n        # Also write the master consolidated CSVs incrementally\n        df_ds_summary.to_csv(summary_csv_path, index=False)\n        df_ds_epochs.to_csv(epochs_csv_path, index=False)\n        df_ds_subcomp.to_csv(subcomp_csv_path, index=False)\n\n        generate_ablation_visualizations(\n            df_all_epochs=df_ds_epochs[df_ds_epochs['dataset'] == ds_name],\n            df_all_subcomp=df_ds_subcomp[df_ds_subcomp['dataset'] == ds_name],\n            df_summary=df_ds_summary[df_ds_summary['dataset'] == ds_name],\n            out_dir=args.out_dir,\n            dataset_name=ds_name\n        )\n\n    total_time = time.time() - total_start\n    print(\"\\n\" + \"#\" * 80)\n    print(\"################### DEC COMPONENT ABLATION SUMMARY ###################\")\n    print(\"#\" * 80)\n    df_final_summary = pd.DataFrame(summary_rows)\n    print(df_final_summary[['dataset', 'ablation', 'seed', 'Initial_DEC_Sil', 'Peak_DEC_Sil', 'Delta_DEC_Sil', 'Peak_DEC_ARI', 'Best_DEC_Epoch']])\n    print(f\"\\n[Done] All logs saved incrementally to:\\n - {summary_csv_path}\\n - {epochs_csv_path}\\n - {subcomp_csv_path}\")\n    print(f\"Per-dataset epoch logs also saved to: {args.out_dir}/<dataset_name>_dec_ablation_epoch_dynamics.csv\")\n    print(f\"Total Execution Time: {total_time/60:.2f} min ({total_time:.1f}s)\")\n\nif __name__ == '__main__':\n    main()\n"

with open("ARISE_Plus_v2_DEC_Ablation.py", "w") as f:
    f.write(ablation_code)

print("✅ Initialized ARISE_Plus_v2_DEC_Ablation.py with full Ablation, Incremental Logging & Re-plotting Suite")


In [ ]:
# @title 🎛️ Step 3: Run DEC Component Ablation Benchmark
DATASET = "0" #@param ["0", "1", "2", "3", "4", "5", "all", "0,1", "4,5"]
ABLATIONS = "all" #@param ["all", "Full_ARISE_v2,wo_Potts_MRF,wo_Sinkhorn_OT", "Full_ARISE_v2,wo_Potts_MRF,wo_Dense_Gram,wo_Spatial_Contrastive", "Full_ARISE_v2,Frozen_Encoder_DEC_Only"]
SEEDS = "42" #@param {type:"string"}
EPOCHS = 400 #@param {type:"integer"}
PRETRAIN_EPOCHS = 250 #@param {type:"integer"}
DEC_EPOCHS = 150 #@param {type:"integer"}
LEARNING_RATE = 0.001 #@param {type:"number"}
OUTPUT_DIR = "results_dec_ablation" #@param {type:"string"}

print(f"Starting DEC Ablation: Dataset={DATASET} | Ablations={ABLATIONS} | Seeds={SEEDS} | Pretrain={PRETRAIN_EPOCHS} | DEC={DEC_EPOCHS}")

!python ARISE_Plus_v2_DEC_Ablation.py \
    --dataset "{DATASET}" \
    --ablations "{ABLATIONS}" \
    --seeds {SEEDS} \
    --epochs {EPOCHS} \
    --pretrain_epochs {PRETRAIN_EPOCHS} \
    --finetune_epochs {DEC_EPOCHS} \
    --lr {LEARNING_RATE} \
    --out_dir "{OUTPUT_DIR}"

In [ ]:
# @title 📊 Step 4: Display Ablation Summary & Silhouette Delta Rankings
import os
import pandas as pd
from IPython.display import display, HTML

summary_path = os.path.join(OUTPUT_DIR, "dec_ablation_summary.csv")
if os.path.exists(summary_path):
    df = pd.read_csv(summary_path)
    print("\n" + "="*70 + "\n  📋 DEC COMPONENT ABLATION SUMMARY & SILHOUETTE GAINS\n" + "="*70)
    display_cols = ['dataset', 'ablation', 'seed', 'Initial_DEC_Sil', 'Peak_DEC_Sil', 'Delta_DEC_Sil', 'Peak_DEC_ARI', 'Best_DEC_Epoch', 'Runtime_Sec']
    display_cols = [c for c in display_cols if c in df.columns]
    display(df[display_cols].sort_values(by='Delta_DEC_Sil', ascending=False))
    
    print("\n" + "="*70 + "\n  🏆 COMPONENT RANKING BY NET SILHOUETTE GAIN (Δ Silhouette)\n" + "="*70)
    ranking = df.groupby('ablation').agg({
        'Initial_DEC_Sil': 'mean',
        'Peak_DEC_Sil': 'mean',
        'Delta_DEC_Sil': 'mean',
        'Peak_DEC_ARI': 'mean',
        'Runtime_Sec': 'mean'
    }).sort_values(by='Delta_DEC_Sil', ascending=False).round(4)
    display(ranking)
else:
    print(f"No summary CSV found at {summary_path}. Please execute Step 3 first.")

In [ ]:
# @title 🖼️ Step 5: Visualize Comparative Trajectories & Dynamics
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

plot_files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.png")))
if len(plot_files) == 0:
    print("No ablation plots found. Run Step 3 first.")
else:
    for pfile in plot_files:
        print(f"\n{'='*70}\nDisplaying: {os.path.basename(pfile)}\n{'='*70}")
        img = mpimg.imread(pfile)
        plt.figure(figsize=(12, 6), dpi=120)
        plt.imshow(img)
        plt.axis('off')
        plt.show()

In [ ]:
# @title 🔄 Step 5b: Re-generate or Customize Plots from Saved CSV Logs Anytime (No Re-training Required)
# You can run this anytime to regenerate all trajectory and ablation charts directly from the saved CSV logs:
!python ARISE_Plus_v2_DEC_Ablation.py --plot_only --out_dir "{OUTPUT_DIR}"

# Or inspect the epoch DataFrame directly in Python:
import os, pandas as pd
epochs_df = pd.read_csv(os.path.join(OUTPUT_DIR, "dec_ablation_epoch_dynamics.csv"))
print(f"Loaded {len(epochs_df)} epoch records across datasets: {epochs_df['dataset'].unique()}")
epochs_df.head(10)


In [ ]:
# @title 📦 Step 6: Download Results Archive (.zip)
from google.colab import files
import shutil

zip_name = f"{OUTPUT_DIR}_archive.zip"
if os.path.exists(OUTPUT_DIR):
    shutil.make_archive(OUTPUT_DIR, 'zip', OUTPUT_DIR)
    files.download(f"{OUTPUT_DIR}.zip")
    print(f"✅ Initiated download for {OUTPUT_DIR}.zip")
else:
    print(f"Directory {OUTPUT_DIR} does not exist.")